## VS Code 로컬 실행

프로젝트 루트 또는 capstone/ 폴더에서 실행합니다. 입력 데이터와 의존성은 README.md를 참고하세요.

현재 출력은 2026년 로컬 실행에서 다시 생성했습니다. 원본 노트북과 당시 저장 출력은 originals/에 보존되어 있습니다.


In [1]:
import pandas as pd
from pathlib import Path
import numpy as np
from numpy import dot
from numpy.linalg import norm

In [ ]:
from pathlib import Path

DATA_CANDIDATES = (
    Path("books_review_2.csv"),
    Path("capstone") / "books_review_2.csv",
)
DATA_PATH = next((path for path in DATA_CANDIDATES if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "books_review_2.csv not found. Run the notebook from the project root "
        "or the capstone/ directory."
    )

df = pd.read_csv(DATA_PATH)
df.head(5)

# 사용자 기반 협업 필터링

참고문헌: 

1) https://towardsdatascience.com/build-a-user-based-collaborative-filtering-recommendation-engine-for-anime-92d35921f304?gi=075be73ac79a


2) https://grabngoinfo.com/recommendation-system-user-based-collaborative-filtering/

3) https://www.geeksforgeeks.org/user-based-collaborative-filtering/


피어슨 상관 계수

In [3]:
print('사용자', df['user_num'].nunique())
print('책 제목', df['Title'].nunique())
print('평점의 개수', df['review/score'].nunique())
print('평점', sorted(df['review/score'].unique()))

사용자 5070
책 제목 1616
평점의 개수 5
평점 [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]


In [4]:
user_matrix =df.pivot_table('review/score', index='user_num', columns='Title') # 사용자 - 책 행렬

In [ ]:
# 사용자 - 책 행렬 정규화

user_matrix_norm = user_matrix.subtract(user_matrix.mean(axis=1), axis = 'rows')
user_matrix_norm.head()

# 피어슨 상관계수

In [6]:
user_similarity_pearson =user_matrix_norm.T.corr() # 피어슨 상관 계수를 사용하여 사용자 유사도 구함

In [ ]:
df[df['user_num']==1225]['categories'].value_counts() # 총 64개

In [8]:
ex_userid =1225 # 사용자 선택
user_similarity_pearson.drop(index=ex_userid, inplace=True) # 선택된 사용자를 삭제

user_based_cf.ipynb cell 12:2: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  user_similarity_pearson.drop(index=ex_userid, inplace=True) # 선택된 사용자를 삭제


In [9]:
n = 10
sim_users = user_similarity_pearson[user_similarity_pearson[ex_userid]>0.3][ex_userid].sort_values(ascending=False)
top_sim_users = sim_users[:n] 

In [10]:
#print(f'{ex_userid}와 유사한 10명의 사용자들 :\n ', top_sim_users)
print('유사한 10명의 사용자들 :\n ', top_sim_users)

유사한 10명의 사용자들 :
  user_num
52      1.0
2178    1.0
4364    1.0
4037    1.0
3560    1.0
3369    1.0
3329    1.0
3198    1.0
3113    1.0
3008    1.0
Name: 1225, dtype: float64


In [11]:
ex_user_books = user_matrix_norm[user_matrix_norm.index == ex_userid].dropna(axis=1, how='all') # 선택한 사용자가 읽은 책들 

In [12]:
top_sim_users_book = user_matrix_norm[user_matrix_norm.index.isin(top_sim_users.index)].dropna(axis=1, how='all') # 비슷한 사용자들이 읽은 책들

In [13]:
top_sim_users_book.drop(ex_user_books.columns,axis=1, inplace=True, errors='ignore') # 사용자가 읽은 책들 제거 

user_based_cf.ipynb cell 17:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  top_sim_users_book.drop(ex_user_books.columns,axis=1, inplace=True, errors='ignore') # 사용자가 읽은 책들 제거


In [14]:
sim_score = {}

for book in top_sim_users_book.columns:
  book_score = top_sim_users_book[book]
  weighted_total = 0.0
  similarity_total = 0.0
  for user in top_sim_users.index:
    if pd.notna(book_score[user]):
      similarity = float(top_sim_users[user])
      weighted_total += similarity * book_score[user]
      similarity_total += abs(similarity)
  if similarity_total > 0:
    sim_score[book] = weighted_total / similarity_total

sim_score = pd.DataFrame(sim_score.items(), columns=['book', 'score'])

sim_score_list = sim_score.sort_values(by='score', ascending=False)
sim_score_list.head(10)

,book,score
10,Ever Since Darwin: Reflections in Natural History,1.500000
37,Zen Guitar,0.750000
23,Sharpe's Eagle: Richard Sharpe and the Talavar...,0.666667
4,Black Edelweiss: A Memoir of Combat and Consci...,0.500000
1,"A Sailor of Austria: In Which, Without Really ...",0.500000
22,Plato: Phaedo,0.437500
18,Nightshade,0.437500
11,Hamlet,0.437500
12,"Heir to the Shadows (The Black Jewels Trilogy,...",0.437500
32,The Painted Kiss: A Novel,0.437500


In [15]:
avg_rating = user_matrix[user_matrix.index == ex_userid].T.mean()[ex_userid] # 선택한 사용자에 대한 평균 평점

print(f'The average movie rating for user {ex_userid} is {avg_rating:.2f}')

The average movie rating for user 1225 is 3.58


In [16]:
# 가중 평균을 통한 예측

sim_score_list['predicted score'] = (sim_score_list['score'] + avg_rating).clip(1, 5)
sim_score_list = sim_score_list.drop(columns=['score'])
sim_score_list.head(10)

,book,predicted score
10,Ever Since Darwin: Reflections in Natural History,5.000000
37,Zen Guitar,4.330645
23,Sharpe's Eagle: Richard Sharpe and the Talavar...,4.247312
4,Black Edelweiss: A Memoir of Combat and Consci...,4.080645
1,"A Sailor of Austria: In Which, Without Really ...",4.080645
22,Plato: Phaedo,4.018145
18,Nightshade,4.018145
11,Hamlet,4.018145
12,"Heir to the Shadows (The Black Jewels Trilogy,...",4.018145
32,The Painted Kiss: A Novel,4.018145
